In [1]:
import h5py
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split


In [ ]:
# ==========================================
# 1. CARICAMENTO DATI DAL FILE HDF5
# ==========================================
h5_filename = '../data/processed/eeg_dataset_v2.h5'

print("Caricamento dataset in memoria...")
with h5py.File(h5_filename, 'r') as hf:
    X_data = hf['X'][:]       # Shape originale: (N, 62, 1500)
    y_meta = hf['y'][:]       # Shape originale: (N, 4)

In [3]:
# ==========================================
# 2. PREPROCESSING
# ==========================================
# Estraiamo solo le etichette delle emozioni (Indice 2 dei metadati: [Paziente, Sessione, Label, Score])
y_labels = y_meta[:, 2]

# Le classi sono 5 (0=Disgust, 1=Fear, 2=Sad, 3=Neutral, 4=Happy). Trasformiamole in One-Hot Encoding
y_cat = to_categorical(y_labels, num_classes=5)

# Per usare Conv2D in Keras, l'input deve avere 4 dimensioni: (Batch_size, Altezza, Larghezza, Canali)
# Aggiungiamo un asse fittizio per i "color channels" (come fosse un'immagine in scala di grigi)
X_data = np.expand_dims(X_data, axis=-1)  # Nuova shape: (N, 62, 1500, 1)


# ==========================================
# SUDDIVISIONE 70% TRAIN / 15% VAL / 15% TEST
# ==========================================

# 1. Primo split: Estraiamo il 15% per il TEST. 
# L'altro 85% finisce in variabili temporanee (X_temp, y_temp)
X_temp, X_test, y_temp, y_test, labels_temp, labels_test = train_test_split(
    X_data, y_cat, y_labels, 
    test_size=0.15, 
    random_state=42, 
    stratify=y_labels # Bilanciamo in base alle emozioni originali
)

# 2. Secondo split: Dividiamo il rimanente 85% in Train e Validation.
# Per far sì che il Validation sia il 15% del totale ORIGINALE, 
# dobbiamo calcolare la frazione rispetto all'85% rimasto (15 / 85 = 0.17647)
val_fraction = 0.15 / 0.85 

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, 
    test_size=val_fraction, 
    random_state=42, 
    stratify=labels_temp # Bilanciamo usando le etichette rimaste in temp
)

print(f"Dimensioni X_train (70%): {X_train.shape}")
print(f"Dimensioni X_val   (15%): {X_val.shape}")
print(f"Dimensioni X_test  (15%): {X_test.shape}")

Dimensioni X_train (70%): (10283, 62, 1500, 1)
Dimensioni X_val   (15%): (2204, 62, 1500, 1)
Dimensioni X_test  (15%): (2204, 62, 1500, 1)


In [4]:
# ==========================================
# 3. DEFINIZIONE ARCHITETTURA CNN 2D (LIGHTWEIGHT)
# ==========================================
model = Sequential(name="EEG_ConvNet2D_Light")

# --- Livello 1: Convoluzione Temporale Leggera ---
# Meno filtri (8) e un kernel più piccolo.
model.add(Conv2D(filters=8, kernel_size=(1, 32), activation='relu', padding='same', input_shape=(62, 1500, 1)))
# POOLING AGGRESSIVO: Riduciamo l'asse del tempo di 10 volte! (da 1500 passa subito a 150)
model.add(MaxPooling2D(pool_size=(1, 10))) 
model.add(Dropout(0.2))

# --- Livello 2: Convoluzione Spaziale ---
# Fonde i 62 canali, meno filtri (16)
model.add(Conv2D(filters=16, kernel_size=(62, 1), activation='relu', padding='valid'))
# Altro pooling per ridurre i 150 campioni rimasti a 30
model.add(MaxPooling2D(pool_size=(1, 5))) 
model.add(Dropout(0.2))

# Abbiamo rimosso il Livello 3 per alleggerire la rete.

# --- Classificatore ---
model.add(Flatten())
model.add(Dense(32, activation='relu')) # Ridotto da 128 a 32
model.add(Dense(5, activation='softmax'))

# Compilazione del modello
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

c:\Users\lucag\miniconda3\envs\eeg-env\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "EEG_ConvNet2D_Light"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 62, 1500, 8)    │           264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 62, 150, 8)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 62, 150, 8)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 1, 150, 16)     │         7,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 1, 30, 16)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1, 30, 16)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 480)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │        15,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           165 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,773 (92.86 KB)

 Trainable params: 23,773 (92.86 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# Addestramento usando X_val e y_val
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_val, y_val) # <--- Inserisci qui il Validation set
)



Epoch 1/50
322/322 ━━━━━━━━━━━━━━━━━━━━ 75s 187ms/step - accuracy: 0.2606 - loss: 1.5981 - val_accuracy: 0.2609 - val_loss: 1.5958
Epoch 2/50
 19/322 ━━━━━━━━━━━━━━━━━━━━ 55s 183ms/step - accuracy: 0.2683 - loss: 1.5869

KeyboardInterrupt: 

In [ ]:
# Valutazione finale (alla cieca) usando X_test e y_test
print("\nValutazione finale sul Test Set:")
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"Accuratezza Test: {test_accuracy * 100:.2f}%")